# 10장 — 🎯 캡스톤: 사내 FAQ 검색봇

7부에서 배운 전부로 **동작하는 FAQ 검색봇**을 조립합니다 — 인덱싱 → 검색 → 그라운딩 → 답변 → 채점.

**이 노트북의 목표**
1. 오프라인 toy 봇을 처음부터 끝까지 조립하고 자동 채점(recall + 정직성)까지
2. (선택) **로컬 Ollama가 있다면** — 진짜 LLM에게 그라운딩 프롬프트를 넘겨 실전 답변 받아 보기

> Ollama가 없어도 전 과정이 돌아갑니다 (실전 셀은 자동으로 건너뜀 — 폴백 완비).

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch10')

## 1. 인덱싱 파이프라인 — 조항 청킹 + 제목 + 정규화 임베딩 (7·9장)

toy 임베딩(주제 축 10개)을 씁니다 — 실전에선 이 함수 하나만 진짜 모델로 바꾸면 됩니다.

In [ ]:
import numpy as np

AXES = ['결혼', '장례', '카드', '연차', '재택', '보안', '동호회', '교육', '절차', '행사']
def embed(**topic_weights):
    """toy 임베딩 — 뜻을 좌표로, 정규화까지 (실전: sentence-transformers 등으로 교체)."""
    v = np.zeros(len(AXES))
    for t, w in topic_weights.items():
        v[AXES.index(t)] = w
    return v / np.linalg.norm(v)

DB = [  # 조항 청킹 + 제목 붙이기 완료 상태
    {'id': 1, 'title': '경조사 규정 > 제1조(결혼)', 'text': '결혼 축하금 50만 원을 지급한다. 단, 근속 1년 이상.', 'vec': embed(결혼=1, 절차=0.1)},
    {'id': 2, 'title': '경조사 규정 > 제2조(장례)', 'text': '장례 지원금 100만 원과 상조 서비스를 지원한다.', 'vec': embed(장례=1)},
    {'id': 3, 'title': '경비 규정 > 제3조(법인카드)', 'text': '법인카드 사용분은 매월 5일까지 경비 시스템에 등록한다.', 'vec': embed(카드=1, 절차=0.2)},
    {'id': 4, 'title': '인사 규정 > 제7조(연차)', 'text': '연차·반차는 인사 포털에서 결재선을 지정해 신청한다.', 'vec': embed(연차=1, 절차=0.3)},
    {'id': 5, 'title': '인사 규정 > 제9조(재택근무)', 'text': '재택근무는 주 2회까지, 팀장 사전 승인으로 운영한다.', 'vec': embed(재택=1, 연차=0.15, 절차=0.2)},
    {'id': 6, 'title': '보안 규정 > 제4조(VPN)', 'text': 'VPN 오류 코드 403은 사내망 인증서 갱신으로 해결한다.', 'vec': embed(보안=1)},
    {'id': 7, 'title': '복지 규정 > 제5조(동호회)', 'text': '동호회는 분기별 20만 원의 활동비를 지원한다.', 'vec': embed(동호회=1)},
    {'id': 8, 'title': '교육 규정 > 제2조(외부 교육)', 'text': '외부 교육은 연 100만 원 한도, 사전 승인 후 수강한다.', 'vec': embed(교육=1, 절차=0.2)},
]
print(f'인덱싱 완료: 조각 {len(DB)}개 — 8건 규모면 벡터 DB 없이 배열 + 내적으로 충분 (6장)')

## 2. 질의 파이프라인 — 검색 → 바닥 점검 → 그라운딩 → 답변

In [ ]:
GROUNDING = ('아래 사내 문서에 근거해서만 답하세요.\n'
             '문서에 없는 내용은 "문서에서 찾지 못했습니다"라고 답하세요.\n'
             '답변 끝에 참고한 문서 제목을 표기하세요.')

def search(q_vec, k=3):
    """코사인(=정규화 내적, 2장) top-k — 전수조사 kNN (6장)."""
    ranked = sorted(DB, key=lambda d: -float(d['vec'] @ q_vec))
    return [(float(d['vec'] @ q_vec), d) for d in ranked[:k]]

def build_prompt(q_text, top):
    docs = '\n'.join(f"[문서{i+1}] {d['title']} — {d['text']}" for i, (_, d) in enumerate(top))
    return f'{GROUNDING}\n\n{docs}\n\n[질문] {q_text}'

def faq_bot(q_text, q_vec, floor=0.5, llm=None):
    """완성된 봇 — llm 함수를 주면 진짜 생성, 없으면 1위 조각 기반 흉내."""
    top = search(q_vec)
    if top[0][0] < floor:  # 바닥 점검 (4장) → 정직한 모름 (8장)
        return top, '문서에서 찾지 못했습니다. 담당 부서에 문의해 주세요.'
    prompt = build_prompt(q_text, top)
    if llm:
        return top, llm(prompt)
    best = top[0][1]
    return top, f"{best['text']} (출처: {best['title']})"

# 시운전 3건 — 정상 / 표현이 다른 질문(워밍업의 그 질문!) / 문서에 없는 질문
tests = [
    ('결혼 축하금은 얼마인가요?', embed(결혼=0.95, 절차=0.15)),
    ('쉬는 날 미리 쓰려면 어디서 해요?', embed(연차=0.85, 절차=0.3)),
    ('올해 겨울 워크숍은 언제인가요?', embed(행사=1, 절차=0.15)),
]
for q_text, q_vec in tests:
    top, ans = faq_bot(q_text, q_vec)
    print(f'🙋 {q_text}')
    print(f'   1위: {top[0][1]["title"]} ({top[0][0]:.2f})')
    print(f'🤖 {ans}\n')

assert '50만 원' in faq_bot(*tests[0])[1] and '근속 1년' in faq_bot(*tests[0])[1], '조건까지 온전한 답 (7장 조항 청킹)'
assert '인사 포털' in faq_bot(*tests[1])[1], '겹치는 단어 없는 질문도 의미로 찾는다 (워밍업의 약속)'
assert '찾지 못했습니다' in faq_bot(*tests[2])[1], '문서에 없으면 정직한 모름 (4·8장)'
print('✔ 시운전 통과 — 아는 것은 출처와 함께, 모르는 것은 모른다고')

## 3. 자동 채점 — 만들었으면 성적표 (7장)

In [ ]:
EVAL = [  # (질문, toy 임베딩, 정답 문서 id)
    ('결혼 축하금은 얼마인가요?', embed(결혼=0.95, 절차=0.15), 1),
    ('쉬는 날 미리 쓰려면 어디서 해요?', embed(연차=0.85, 절차=0.3), 4),
    ('법인카드 정산 마감이 언제죠?', embed(카드=0.9, 절차=0.25), 3),
    ('재택은 일주일에 며칠까지 되나요?', embed(재택=0.9, 연차=0.1), 5),
    ('외부 강의 수강하면 지원 되나요?', embed(교육=0.88, 절차=0.2), 8),
]

def recall_at(k):
    hits = 0
    for q_text, q_vec, expect in EVAL:
        ranked_ids = [d['id'] for _, d in search(q_vec, k=k)]
        hits += int(expect in ranked_ids)
    return hits / len(EVAL)

print(f'recall@1 = {recall_at(1):.0%}   recall@3 = {recall_at(3):.0%}')
assert recall_at(1) == 1.0, 'toy 지도는 만점 — 실전에서는 수십~수백 문항으로 재고 다듬는 반복(7장)'
print('✔ 성적표 완료 — 실전 개선 사이클: 로그로 평가 세트 확장 → recall 측정 → 청킹·임베딩·k 개선')

## 4. (선택) 실전 맛보기 — 로컬 Ollama에게 진짜 생성 맡기기

그라운딩 프롬프트를 **조련된 LLM**(6부)에게 넘깁니다. Ollama가 안 떠 있으면 자동으로 건너뜁니다.

In [ ]:
import requests

def ollama_llm(prompt, url='http://localhost:11434', model='llama3.2'):
    r = requests.post(f'{url}/api/generate', json={'model': model, 'prompt': prompt, 'stream': False}, timeout=60)
    r.raise_for_status()
    return r.json()['response'].strip()

try:
    requests.get('http://localhost:11434', timeout=2)  # 서버 확인
    q_text, q_vec = tests[0]
    top, ans = faq_bot(q_text, q_vec, llm=ollama_llm)
    print(f'🙋 {q_text}')
    print(f'🤖 (진짜 LLM) {ans}')
    print('\n✔ 방금 완전한 실전 RAG가 돌았습니다 — toy 임베딩만 진짜 모델로 바꾸면 끝!')
except Exception:
    print('ℹ️ Ollama가 실행 중이 아니라 이 셀은 건너뜁니다 (학습에는 지장 없음).')
    print('   실전 전환 요령: ① embed()를 임베딩 모델로 ② DB를 pgvector·Chroma로')
    print('   ③ faq_bot(..., llm=ollama_llm)으로 진짜 생성 — 구조는 오늘 그대로.')

## 5. 🎉 7부 완주 — 여정 정리

**"의미는 위치다"** 한 문장에서 출발해 —

| 비유 | 실무 용어 | 장 |
|---|---|---|
| 의미 지도 | 임베딩 공간 (embedding space) | 워밍업·3장 |
| 가까움 재는 자 | 코사인 유사도 (정규화 시 내적) | 2장 |
| 무작위의 바닥 | 고차원 집중 (0 ± 1/√d) | 4장 |
| 조각 / 칼질 | 청크 / 청킹 (chunking) | 7장 |
| 창고 + 색인 | 벡터 DB + ANN (IVF·HNSW) | 6장 |
| 검색 성적표 | recall@k | 7장 |
| 답을 문서에 묶기 | 그라운딩 (grounding) vs 환각 | 8장 |
| 짝은 가깝게 남은 멀게 | 대조 학습 (InfoNCE) | 9장 |
| 지식 주입의 두 길 | 파인튜닝(6부) vs RAG(검색 증강 생성) | 1·8장 |

— 봇 한 대에 시리즈 전체(1부 내적·2부 경사하강·3부 집중과 신뢰구간·4부 임베딩과 온도·
5부 −log·6부 조련과 평가)가 들어 있습니다.

다음 걸음: 회사의 진짜 문서 10건으로 시작해 보세요 — 구조는 그대로, 부품 등급만 바꾸면 됩니다. 🚀